# OpenMC GitHub Issues Analysis

This notebook provides statistical and visual analysis of GitHub issues from the OpenMC repository.

It analyzes:
- Issue variety and distribution by type
- Issue classes (bug, feature, question, etc.)
- Complexity levels
- Temporal patterns
- Response characteristics

## Prerequisites

First, run the validation script to generate the data:

```bash
pip install -r tools/ai/requirements.txt
export GITHUB_TOKEN=your_token
python tools/ai/validate_issue_responder.py --repo openmc-dev/openmc --max-issues 100
```

In [ ]:
# Import required libraries
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ Libraries imported successfully")

## Load Validation Results

In [ ]:
# Load the validation results
results_path = Path('test_results/validation_results_latest.json')

if not results_path.exists():
    print("❌ Results file not found. Please run the validation script first:")
    print("   python tools/ai/validate_issue_responder.py --repo openmc-dev/openmc")
else:
    with open(results_path, 'r') as f:
        results = json.load(f)
    
    # Convert to DataFrame
    df = pd.DataFrame(results['test_cases'])
    
    print(f"✓ Loaded {len(df)} test cases")
    print(f"  Total cases: {results['total_cases']}")
    print(f"  Successful: {results['successful']}")
    print(f"  Failed: {results['failed']}")
    
    # Display first few rows
    df.head()

## Data Overview

In [ ]:
# Basic statistics
print("Dataset Overview:")
print("=" * 60)
print(f"Total Issues: {len(df)}")
print(f"\nIssue States:")
print(df['state'].value_counts())
print(f"\nIssue Types:")
print(df['predicted_type'].value_counts())
print(f"\nComplexity Levels:")
print(df['predicted_complexity'].value_counts())
print(f"\nAverage Body Length: {df['body_length'].mean():.0f} characters")
print(f"Average Comments Count: {df['comments_count'].mean():.1f}")
print(f"Average Processing Time: {df['processing_time'].mean():.3f}s")

## 1. Issue Type Distribution

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart of issue types
type_counts = df['predicted_type'].value_counts()
colors = sns.color_palette('husl', len(type_counts))
axes[0].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%', 
            colors=colors, startangle=90)
axes[0].set_title('Issue Type Distribution', fontsize=14, fontweight='bold')

# Bar chart of issue types
type_counts.plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_title('Issue Type Counts', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Issue Type', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('test_results/issue_type_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: test_results/issue_type_distribution.png")

## 2. Complexity Analysis

In [ ]:
# Complexity distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Overall complexity distribution
complexity_counts = df['predicted_complexity'].value_counts()
complexity_order = ['trivial', 'simple', 'moderate', 'complex', 'very_complex']
complexity_counts = complexity_counts.reindex(complexity_order, fill_value=0)

axes[0, 0].bar(range(len(complexity_counts)), complexity_counts.values, 
               color=sns.color_palette('RdYlGn_r', len(complexity_counts)))
axes[0, 0].set_xticks(range(len(complexity_counts)))
axes[0, 0].set_xticklabels(complexity_counts.index, rotation=45)
axes[0, 0].set_title('Overall Complexity Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Count', fontsize=12)
axes[0, 0].grid(axis='y', alpha=0.3)

# Complexity by issue type (heatmap)
complexity_by_type = pd.crosstab(df['predicted_type'], df['predicted_complexity'])
complexity_by_type = complexity_by_type.reindex(columns=complexity_order, fill_value=0)
sns.heatmap(complexity_by_type, annot=True, fmt='d', cmap='YlOrRd', ax=axes[0, 1], cbar_kws={'label': 'Count'})
axes[0, 1].set_title('Complexity by Issue Type', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Complexity', fontsize=12)
axes[0, 1].set_ylabel('Issue Type', fontsize=12)

# Body length vs complexity
complexity_to_num = {'trivial': 1, 'simple': 2, 'moderate': 3, 'complex': 4, 'very_complex': 5}
df['complexity_num'] = df['predicted_complexity'].map(complexity_to_num)
axes[1, 0].scatter(df['body_length'], df['complexity_num'], alpha=0.6, s=50)
axes[1, 0].set_xlabel('Body Length (characters)', fontsize=12)
axes[1, 0].set_ylabel('Complexity Level', fontsize=12)
axes[1, 0].set_yticks([1, 2, 3, 4, 5])
axes[1, 0].set_yticklabels(['Trivial', 'Simple', 'Moderate', 'Complex', 'Very Complex'])
axes[1, 0].set_title('Issue Body Length vs Complexity', fontsize=14, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Processing time by complexity
df.boxplot(column='processing_time', by='predicted_complexity', ax=axes[1, 1])
axes[1, 1].set_title('Processing Time by Complexity', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Complexity', fontsize=12)
axes[1, 1].set_ylabel('Processing Time (seconds)', fontsize=12)
axes[1, 1].tick_params(axis='x', rotation=45)
plt.suptitle('')  # Remove the automatic title

plt.tight_layout()
plt.savefig('test_results/complexity_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: test_results/complexity_analysis.png")

## 3. Temporal Analysis

In [ ]:
# Convert created_at to datetime
df['created_at'] = pd.to_datetime(df['created_at'])
df['created_date'] = df['created_at'].dt.date
df['created_month'] = df['created_at'].dt.to_period('M')

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Issues over time
issues_by_month = df.groupby('created_month').size()
axes[0].plot(issues_by_month.index.to_timestamp(), issues_by_month.values, marker='o', linewidth=2)
axes[0].set_title('Issues Created Over Time', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Month', fontsize=12)
axes[0].set_ylabel('Number of Issues', fontsize=12)
axes[0].grid(alpha=0.3)

# Issue types over time (stacked)
issues_by_month_type = df.groupby(['created_month', 'predicted_type']).size().unstack(fill_value=0)
issues_by_month_type.plot(kind='area', stacked=True, ax=axes[1], alpha=0.7)
axes[1].set_title('Issue Types Over Time (Stacked)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Month', fontsize=12)
axes[1].set_ylabel('Number of Issues', fontsize=12)
axes[1].legend(title='Issue Type', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('test_results/temporal_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: test_results/temporal_analysis.png")

## 4. Label Analysis

In [ ]:
# Flatten labels and count
all_labels = []
for labels in df['labels']:
    all_labels.extend(labels)

label_counts = Counter(all_labels)
top_labels = dict(sorted(label_counts.items(), key=lambda x: x[1], reverse=True)[:15])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top labels bar chart
axes[0].barh(list(top_labels.keys()), list(top_labels.values()), color=sns.color_palette('viridis', len(top_labels)))
axes[0].set_title('Top 15 Issue Labels', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Count', fontsize=12)
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# Issues with/without labels
df['has_labels'] = df['labels'].apply(lambda x: 'Has Labels' if len(x) > 0 else 'No Labels')
label_status = df['has_labels'].value_counts()
axes[1].pie(label_status.values, labels=label_status.index, autopct='%1.1f%%', 
            colors=['#66c2a5', '#fc8d62'], startangle=90)
axes[1].set_title('Issues with vs without Labels', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('test_results/label_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Total unique labels: {len(label_counts)}")
print(f"✓ Issues with labels: {(df['labels'].apply(len) > 0).sum()}")
print("✓ Saved: test_results/label_analysis.png")

## 5. Response Characteristics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Response length distribution
axes[0, 0].hist(df['response_length'], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df['response_length'].mean(), color='red', linestyle='--', 
                   linewidth=2, label=f'Mean: {df["response_length"].mean():.0f}')
axes[0, 0].set_title('Response Length Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Response Length (characters)', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Response length by issue type
df.boxplot(column='response_length', by='predicted_type', ax=axes[0, 1])
axes[0, 1].set_title('Response Length by Issue Type', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Issue Type', fontsize=12)
axes[0, 1].set_ylabel('Response Length (characters)', fontsize=12)
axes[0, 1].tick_params(axis='x', rotation=45)
plt.suptitle('')

# Processing time distribution
axes[1, 0].hist(df['processing_time'], bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(df['processing_time'].mean(), color='darkred', linestyle='--', 
                   linewidth=2, label=f'Mean: {df["processing_time"].mean():.3f}s')
axes[1, 0].set_title('Processing Time Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Processing Time (seconds)', fontsize=12)
axes[1, 0].set_ylabel('Frequency', fontsize=12)
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Processing time vs body length
axes[1, 1].scatter(df['body_length'], df['processing_time'], alpha=0.6, s=50, c=df['complexity_num'], 
                   cmap='RdYlGn_r')
axes[1, 1].set_title('Processing Time vs Body Length', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Body Length (characters)', fontsize=12)
axes[1, 1].set_ylabel('Processing Time (seconds)', fontsize=12)
axes[1, 1].grid(alpha=0.3)
cbar = plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1])
cbar.set_label('Complexity', rotation=270, labelpad=20)

plt.tight_layout()
plt.savefig('test_results/response_characteristics.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: test_results/response_characteristics.png")

## 6. Classification Accuracy

In [ ]:
# Check accuracy where expected type is available
df_with_expected = df[df['expected_type'].notna()].copy()

if len(df_with_expected) > 0:
    # Calculate accuracy
    df_with_expected['correct'] = df_with_expected['expected_type'] == df_with_expected['predicted_type']
    accuracy = df_with_expected['correct'].mean() * 100
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Accuracy pie chart
    accuracy_counts = df_with_expected['correct'].value_counts()
    axes[0].pie(accuracy_counts.values, labels=['Correct', 'Incorrect'], 
                autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
    axes[0].set_title(f'Overall Classification Accuracy: {accuracy:.1f}%', 
                      fontsize=14, fontweight='bold')
    
    # Accuracy by issue type
    accuracy_by_type = df_with_expected.groupby('expected_type')['correct'].agg(['sum', 'count'])
    accuracy_by_type['accuracy'] = (accuracy_by_type['sum'] / accuracy_by_type['count']) * 100
    accuracy_by_type['accuracy'].plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2'))
    axes[1].set_title('Classification Accuracy by Issue Type', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Expected Issue Type', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].set_ylim(0, 100)
    axes[1].grid(axis='y', alpha=0.3)
    axes[1].axhline(y=accuracy, color='red', linestyle='--', label='Overall')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('test_results/classification_accuracy.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Overall Accuracy: {accuracy:.1f}%")
    print(f"✓ Correct: {df_with_expected['correct'].sum()}")
    print(f"✓ Incorrect: {(~df_with_expected['correct']).sum()}")
    print("✓ Saved: test_results/classification_accuracy.png")
else:
    print("⚠ No issues with expected types for accuracy calculation")

## 7. Statistical Summary

In [ ]:
# Generate comprehensive statistical summary
print("\n" + "="*80)
print("COMPREHENSIVE STATISTICAL SUMMARY")
print("="*80)

print("\n1. ISSUE TYPE DISTRIBUTION")
print("-" * 80)
type_dist = df['predicted_type'].value_counts()
for itype, count in type_dist.items():
    percentage = (count / len(df)) * 100
    print(f"  {itype.title():20s}: {count:3d} ({percentage:5.1f}%)")

print("\n2. COMPLEXITY DISTRIBUTION")
print("-" * 80)
for complexity in complexity_order:
    count = (df['predicted_complexity'] == complexity).sum()
    percentage = (count / len(df)) * 100
    print(f"  {complexity.replace('_', ' ').title():20s}: {count:3d} ({percentage:5.1f}%)")

print("\n3. RESPONSE METRICS")
print("-" * 80)
print(f"  Response Length:")
print(f"    Mean:     {df['response_length'].mean():8.1f} characters")
print(f"    Median:   {df['response_length'].median():8.1f} characters")
print(f"    Std Dev:  {df['response_length'].std():8.1f} characters")
print(f"    Min:      {df['response_length'].min():8.1f} characters")
print(f"    Max:      {df['response_length'].max():8.1f} characters")

print(f"\n  Processing Time:")
print(f"    Mean:     {df['processing_time'].mean():8.3f} seconds")
print(f"    Median:   {df['processing_time'].median():8.3f} seconds")
print(f"    Std Dev:  {df['processing_time'].std():8.3f} seconds")
print(f"    Min:      {df['processing_time'].min():8.3f} seconds")
print(f"    Max:      {df['processing_time'].max():8.3f} seconds")

print("\n4. ISSUE CHARACTERISTICS")
print("-" * 80)
print(f"  Body Length:")
print(f"    Mean:     {df['body_length'].mean():8.1f} characters")
print(f"    Median:   {df['body_length'].median():8.1f} characters")
print(f"\n  Comments:")
print(f"    Mean:     {df['comments_count'].mean():8.1f} comments")
print(f"    Median:   {df['comments_count'].median():8.1f} comments")
print(f"\n  Labels:")
print(f"    Issues with labels:    {(df['labels'].apply(len) > 0).sum():3d} ({((df['labels'].apply(len) > 0).sum() / len(df) * 100):5.1f}%)")
print(f"    Issues without labels: {(df['labels'].apply(len) == 0).sum():3d} ({((df['labels'].apply(len) == 0).sum() / len(df) * 100):5.1f}%)")
print(f"    Total unique labels:   {len(label_counts):3d}")

print("\n5. TEMPORAL INSIGHTS")
print("-" * 80)
print(f"  Date Range: {df['created_at'].min().date()} to {df['created_at'].max().date()}")
print(f"  Time Span: {(df['created_at'].max() - df['created_at'].min()).days} days")
print(f"  Average Issues per Month: {len(df) / max(1, df['created_month'].nunique()):.1f}")

if len(df_with_expected) > 0:
    print("\n6. CLASSIFICATION ACCURACY")
    print("-" * 80)
    print(f"  Overall Accuracy:      {accuracy:.1f}%")
    print(f"  Correctly Classified:  {df_with_expected['correct'].sum():3d}")
    print(f"  Incorrectly Classified: {(~df_with_expected['correct']).sum():3d}")
    print(f"\n  Accuracy by Type:")
    for itype, row in accuracy_by_type.iterrows():
        print(f"    {itype.title():20s}: {row['accuracy']:5.1f}% ({int(row['sum'])}/{int(row['count'])})")

print("\n" + "="*80)

## 8. Export Summary Report

In [ ]:
# Create a comprehensive summary CSV
summary_stats = {
    'Metric': [],
    'Value': []
}

summary_stats['Metric'].extend(['Total Issues', 'Successful Processing', 'Failed Processing'])
summary_stats['Value'].extend([len(df), results['successful'], results['failed']])

for itype, count in type_dist.items():
    summary_stats['Metric'].append(f'Issues: {itype}')
    summary_stats['Value'].append(count)

summary_stats['Metric'].extend(['Avg Response Length', 'Avg Processing Time', 'Avg Body Length'])
summary_stats['Value'].extend([
    f"{df['response_length'].mean():.1f}",
    f"{df['processing_time'].mean():.3f}s",
    f"{df['body_length'].mean():.1f}"
])

summary_df = pd.DataFrame(summary_stats)
summary_df.to_csv('test_results/analysis_summary.csv', index=False)

# Export detailed issue data
df.to_csv('test_results/detailed_issue_data.csv', index=False)

print("✓ Exported summary_stats to: test_results/analysis_summary.csv")
print("✓ Exported detailed data to: test_results/detailed_issue_data.csv")
print("\nAll visualizations saved to test_results/ directory:")
print("  - issue_type_distribution.png")
print("  - complexity_analysis.png")
print("  - temporal_analysis.png")
print("  - label_analysis.png")
print("  - response_characteristics.png")
if len(df_with_expected) > 0:
    print("  - classification_accuracy.png")

## Conclusion

This notebook provides comprehensive statistical and visual analysis of GitHub issues from the OpenMC repository. The analysis includes:

1. **Issue Type Distribution**: Understanding the variety of issues (bugs, features, questions, etc.)
2. **Complexity Analysis**: Assessing issue complexity levels and their relationship with content length
3. **Temporal Patterns**: Identifying trends in issue creation over time
4. **Label Analysis**: Understanding labeling patterns and coverage
5. **Response Characteristics**: Analyzing the AI responder's performance metrics
6. **Classification Accuracy**: Measuring the accuracy of the automated classification

All visualizations and data exports are saved in the `test_results/` directory for use in reports and presentations.